# Actividad 10. Clasificación con XGBoost

XGBoost es un ensamble de árboles con regularización. Se usa **la misma
partición temporal y las mismas variables** de la red neuronal para que la
comparación sea justa:

* entrenamiento 2000–2022, validación 2023 y prueba 2024–2025;
* 30 configuraciones exploradas con búsqueda aleatoria;
* `early stopping` sobre la ROC-AUC de validación;
* calibración de temperatura y umbral elegidos **solo con validación**.

El conjunto de prueba no participa en ninguna decisión.

## 1. Configuración y artefactos

In [1]:
import json
import math
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados" / "ml"
GRAFICAS = RAIZ / "resultados" / "graficas" / "ml"
TABLAS = RAIZ / "resultados" / "tablas"
for carpeta in (RESULTADOS, GRAFICAS, TABLAS):
    carpeta.mkdir(parents=True, exist_ok=True)

PALETA = {
    "purpura": "#4C1D95",
    "purpura_claro": "#8B5CF6",
    "rosa": "#FA8A9A",
    "amarillo": "#FFB71B",
    "rojo": "#EE3E29",
    "coral": "#FF8A5F",
    "azul": "#1B4576",
    "gris": "#7A7191",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 150,
        "font.family": "DejaVu Sans",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.22,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
    }
)


def guardar(fig, nombre):
    ruta = GRAFICAS / nombre
    fig.savefig(ruta, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Figura guardada: {ruta.relative_to(RAIZ)}")

In [2]:
metricas = json.loads((MODELOS / "metricas.json").read_text(encoding="utf-8"))
clf = metricas["clasificacion"]
nn = clf["red_neuronal"]
xgb = clf["xgboost"]

val_xgb = pd.read_csv(RESULTADOS / "pred_xgb_val.csv")
test_xgb = pd.read_csv(RESULTADOS / "pred_xgb_test.csv")
val_nn = pd.read_csv(RESULTADOS / "pred_nn_val.csv")
test_nn = pd.read_csv(RESULTADOS / "pred_nn_test.csv")
busqueda = pd.read_csv(RESULTADOS / "busqueda_xgb_clasificacion.csv")
shap = pd.read_csv(RESULTADOS / "shap_xgb_clasificacion.csv")
cal_xgb = pd.read_csv(RESULTADOS / "calibracion_xgb.csv")
cal_nn = pd.read_csv(RESULTADOS / "calibracion_nn.csv")

print(f"Configuraciones exploradas: {len(busqueda)}")
print(f"Temperatura de calibración: {clf['temperaturas']['xgboost']:.3f}")
print(f"Umbral elegido en validación: {clf['umbrales']['xgboost']:.2f}")
print(f"Eventos de prueba: {len(test_xgb):,}")

Configuraciones exploradas: 30
Temperatura de calibración: 0.721
Umbral elegido en validación: 0.39
Eventos de prueba: 813


## 2. Búsqueda de hiperparámetros

La búsqueda aleatoria prioriza la ROC-AUC de validación. La mejor
configuración se reutiliza en las actividades de interpretabilidad y en la
aplicación web.

In [3]:
ordenada = busqueda.sort_values("auc_validacion", ascending=False).reset_index(drop=True)
mejor = ordenada.iloc[0]
columnas_mejor = [
    "prueba", "auc_validacion", "mejor_iteracion", "max_depth", "eta",
    "subsample", "colsample_bytree", "min_child_weight", "gamma", "lambda", "alpha",
]
display(ordenada[columnas_mejor].head(8).round(4))
ordenada[columnas_mejor].to_csv(TABLAS / "actividad10_busqueda.csv", index=False)

fig, eje = plt.subplots(figsize=(8.5, 4))
eje.scatter(busqueda["prueba"], busqueda["auc_validacion"], color=PALETA["purpura_claro"], s=45, alpha=0.8)
eje.scatter([mejor["prueba"]], [mejor["auc_validacion"]], color=PALETA["rojo"], s=110, zorder=3, label="Mejor configuración")
eje.axhline(mejor["auc_validacion"], color=PALETA["rojo"], linestyle="--", linewidth=1.2)
eje.set_xlabel("Intento")
eje.set_ylabel("ROC-AUC de validación")
eje.set_title("Búsqueda aleatoria de hiperparámetros de XGBoost")
eje.legend(frameon=False)
guardar(fig, "actividad10_busqueda.png")

print(f"Mejor intento: {int(mejor['prueba'])} · ROC-AUC validación {mejor['auc_validacion']:.4f}")

,prueba,auc_validacion,mejor_iteracion,max_depth,eta,subsample,colsample_bytree,min_child_weight,gamma,lambda,alpha
0,13,0.8198,42,7,0.07,0.7,0.6,1,0.1,1.0,0.5
1,26,0.8170,126,7,0.07,0.8,0.9,2,0.3,5.0,0.0
2,6,0.8144,182,7,0.03,1.0,0.6,8,0.3,2.0,0.5
3,27,0.8135,124,6,0.10,0.7,0.6,8,0.0,0.5,0.0
4,20,0.8117,285,6,0.04,0.8,0.8,5,0.3,5.0,0.5
5,14,0.8104,139,7,0.03,1.0,0.9,8,0.5,2.0,0.5
6,17,0.8103,24,4,0.10,0.7,1.0,3,0.5,0.5,0.0
7,4,0.8096,439,3,0.04,1.0,0.8,3,0.1,1.0,0.1


Figura guardada: resultados\graficas\ml\actividad10_busqueda.png
Mejor intento: 13 · ROC-AUC validación 0.8198


## 3. Métricas de validación y prueba

In [4]:
claves = ["roc_auc", "pr_auc", "f1", "precision", "recall", "brier", "log_loss", "exactitud"]
tabla = pd.DataFrame(
    {
        particion: {clave: xgb[particion][clave] for clave in claves}
        for particion in ("validacion", "prueba")
    }
).rename(columns={"validacion": "Validación 2023", "prueba": "Prueba 2024–2025"})
display(tabla.round(4))
tabla.to_csv(TABLAS / "actividad10_metricas_xgb.csv")

fig, eje = plt.subplots(figsize=(8.5, 4))
x = np.arange(len(claves))
eje.bar(x - 0.19, tabla["Validación 2023"], width=0.38, color=PALETA["amarillo"], label="Validación 2023")
eje.bar(x + 0.19, tabla["Prueba 2024–2025"], width=0.38, color=PALETA["purpura"], label="Prueba 2024–2025")
eje.set_xticks(x)
eje.set_xticklabels(["ROC-AUC", "PR-AUC", "F1", "Precisión", "Recall", "Brier", "Log-loss", "Exactitud"], rotation=25, ha="right")
eje.set_ylim(0, 1)
eje.set_title("XGBoost: métricas por partición")
eje.legend(frameon=False)
guardar(fig, "actividad10_metricas.png")

,Validación 2023,Prueba 2024–2025
roc_auc,0.8198,0.8267
pr_auc,0.6847,0.7710
f1,0.6706,0.7115
precision,0.5990,0.6364
recall,0.7616,0.8069
brier,0.1631,0.1639
log_loss,0.4944,0.5017
exactitud,0.7472,0.7417


Figura guardada: resultados\graficas\ml\actividad10_metricas.png


## 4. Curvas ROC y precisión-recall

Se comparan las curvas de XGBoost y de la red neuronal sobre el mismo
conjunto de prueba. Una curva más cercana a la esquina superior izquierda
indica mejor capacidad de ordenar los eventos.

In [5]:
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.6))
for nombre, datos_parte, columna, color in [
    ("XGBoost · validación", val_xgb, "xgb", PALETA["amarillo"]),
    ("XGBoost · prueba", test_xgb, "xgb", PALETA["purpura"]),
    ("Red neuronal · prueba", test_nn, "nn", PALETA["rojo"]),
]:
    fpr, tpr, _ = roc_curve(datos_parte["y"], datos_parte[columna])
    ejes[0].plot(fpr, tpr, color=color, linewidth=2, label=f"{nombre} · AUC {roc_auc_score(datos_parte['y'], datos_parte[columna]):.3f}")
    precision, recall, _ = precision_recall_curve(datos_parte["y"], datos_parte[columna])
    ejes[1].plot(recall, precision, color=color, linewidth=2, label=f"{nombre} · AP {average_precision_score(datos_parte['y'], datos_parte[columna]):.3f}")

ejes[0].plot([0, 1], [0, 1], linestyle="--", color=PALETA["gris"], linewidth=1)
ejes[0].set_xlabel("Tasa de falsos positivos")
ejes[0].set_ylabel("Tasa de verdaderos positivos")
ejes[0].set_title("Curva ROC")
ejes[0].legend(frameon=False, loc="lower right", fontsize=8)
ejes[1].set_xlabel("Recall")
ejes[1].set_ylabel("Precisión")
ejes[1].set_title("Curva precisión-recall")
ejes[1].legend(frameon=False, loc="lower left", fontsize=8)
guardar(fig, "actividad10_curvas.png")

Figura guardada: resultados\graficas\ml\actividad10_curvas.png


## 5. Calibración

In [6]:
fig, eje = plt.subplots(figsize=(6.4, 5))
eje.plot([0, 1], [0, 1], linestyle="--", color=PALETA["gris"], label="Calibración perfecta")
eje.plot(cal_xgb["probabilidad_media"], cal_xgb["frecuencia_observada"], marker="o", color=PALETA["purpura"], linewidth=2, label="XGBoost")
eje.plot(cal_nn["probabilidad_media"], cal_nn["frecuencia_observada"], marker="s", color=PALETA["rojo"], linewidth=2, label="Red neuronal")
eje.set_xlabel("Probabilidad estimada")
eje.set_ylabel("Frecuencia observada")
eje.set_title("Calibración en prueba 2024–2025")
eje.legend(frameon=False)
guardar(fig, "actividad10_calibracion.png")

Figura guardada: resultados\graficas\ml\actividad10_calibracion.png


## 6. Matriz de confusión

In [7]:
from sklearn.metrics import confusion_matrix

umbral = xgb["prueba"]["umbral"]
matriz = confusion_matrix(test_xgb["y"], (test_xgb["xgb"] >= umbral).astype(int), labels=[0, 1])
fig, eje = plt.subplots(figsize=(5.2, 4.4))
eje.imshow(matriz, cmap="Purples")
for i in range(2):
    for j in range(2):
        eje.text(j, i, f"{matriz[i, j]:,}", ha="center", va="center", fontsize=14, color="white" if matriz[i, j] > matriz.max() / 2 else PALETA["purpura"])
eje.set_xticks([0, 1])
eje.set_yticks([0, 1])
eje.set_xticklabels(["Predice ordinario", "Predice alto"])
eje.set_yticklabels(["Real ordinario", "Real alto"])
eje.set_title(f"XGBoost · umbral {umbral:.2f}")
guardar(fig, "actividad10_matriz.png")

print(f"Verdaderos negativos: {matriz[0, 0]:,}")
print(f"Falsos positivos    : {matriz[0, 1]:,}")
print(f"Falsos negativos    : {matriz[1, 0]:,}")
print(f"Verdaderos positivos: {matriz[1, 1]:,}")

Figura guardada: resultados\graficas\ml\actividad10_matriz.png
Verdaderos negativos: 344
Falsos positivos    : 148
Falsos negativos    : 62
Verdaderos positivos: 259


## 7. ¿Qué variables pesan más?

Se usa el **TreeSHAP nativo de XGBoost**: no requiere el paquete `shap` y
mide la contribución media absoluta de cada variable en la predicción.

In [8]:
from src.ml import config

def nombre_legible(variable):
    if variable.startswith("num__"):
        return config.ETIQUETAS.get(variable[5:], variable[5:])
    if variable.startswith("cat__"):
        resto = variable[6:]
        columna, _, categoria = resto.partition("__")
        return f"{config.ETIQUETAS.get(columna, columna)}: {categoria}"
    return variable

top = shap.head(18).copy()
top["etiqueta"] = top["variable"].map(nombre_legible)
display(top[["etiqueta", "shap_medio_absoluto"]].round(4))
top.to_csv(TABLAS / "actividad10_shap_top.csv", index=False)

fig, eje = plt.subplots(figsize=(8.8, 6))
datos = top.sort_values("shap_medio_absoluto")
eje.barh(datos["etiqueta"], datos["shap_medio_absoluto"], color=PALETA["purpura_claro"])
eje.set_xlabel("Contribución media absoluta (SHAP)")
eje.set_title("Variables más influyentes en XGBoost")
guardar(fig, "actividad10_shap.png")

,etiqueta,shap_medio_absoluto
0,PIB per cápita,0.4022
1,Duración del evento,0.3821
2,Población,0.2009
3,Duración no reportada,0.1495
4,egion: Asia,0.1362
5,Densidad de población,0.1228
6,Magnitud típica,0.1174
7,ubtipo: Tropical cyclone,0.1130
8,ipo: Drought,0.0920
9,ubtipo: Heat wave,0.0764


Figura guardada: resultados\graficas\ml\actividad10_shap.png


## 8. Comparación formal con la red neuronal

No basta comparar dos números: se reporta la diferencia pareada con un
intervalo de confianza bootstrap al 95 %. Si el intervalo incluye el cero, la
diferencia no es concluyente con 813 eventos de prueba.

In [9]:
filas = []
for clave in ["roc_auc", "pr_auc", "f1", "precision", "recall", "brier"]:
    filas.append(
        {
            "metrica": clave,
            "red_neuronal": nn["prueba"][clave],
            "xgboost": xgb["prueba"][clave],
            "diferencia_xgb_menos_red": xgb["prueba"][clave] - nn["prueba"][clave],
        }
    )
comparacion = pd.DataFrame(filas)
display(comparacion.round(4))
comparacion.to_csv(TABLAS / "actividad10_comparacion.csv", index=False)

filas_boot = []
for metrica in ("roc_auc", "pr_auc"):
    datos = clf["comparacion_nn_vs_xgb"][metrica]
    filas_boot.append(
        {
            "metrica": metrica.upper(),
            "diferencia_xgb_menos_red": datos["diferencia"],
            "ic95_inferior": datos["ic95_inferior"],
            "ic95_superior": datos["ic95_superior"],
            "p_valor_aproximado": datos["p_valor_aproximado"],
        }
    )
bootstrap = pd.DataFrame(filas_boot)
display(bootstrap.round(4))
bootstrap.to_csv(TABLAS / "actividad10_bootstrap.csv", index=False)

fig, eje = plt.subplots(figsize=(8.5, 4.2))
x = np.arange(3)
red = [nn["prueba"]["roc_auc"], nn["prueba"]["pr_auc"], nn["prueba"]["f1"]]
arboles = [xgb["prueba"]["roc_auc"], xgb["prueba"]["pr_auc"], xgb["prueba"]["f1"]]
eje.bar(x - 0.19, red, width=0.38, color=PALETA["rojo"], label="Red neuronal")
eje.bar(x + 0.19, arboles, width=0.38, color=PALETA["purpura"], label="XGBoost")
eje.set_xticks(x)
eje.set_xticklabels(["ROC-AUC", "PR-AUC", "F1"])
eje.set_ylim(0, 1)
eje.set_title("Comparación en prueba 2024–2025")
eje.legend(frameon=False)
guardar(fig, "actividad10_comparacion.png")

,metrica,red_neuronal,xgboost,diferencia_xgb_menos_red
0,roc_auc,0.8197,0.8267,0.0071
1,pr_auc,0.7653,0.7710,0.0057
2,f1,0.7044,0.7115,0.0071
3,precision,0.6328,0.6364,0.0036
4,recall,0.7944,0.8069,0.0125
5,brier,0.1678,0.1639,-0.0040


,metrica,diferencia_xgb_menos_red,ic95_inferior,ic95_superior,p_valor_aproximado
0,ROC_AUC,-0.0071,-0.0208,0.0064,0.296
1,PR_AUC,-0.0057,-0.0275,0.0142,0.550


Figura guardada: resultados\graficas\ml\actividad10_comparacion.png


## Conclusión

* XGBoost alcanza **0.827 de ROC-AUC** y **0.712 de F1** en prueba; la red
  neuronal obtiene **0.820** y **0.704**.
* La diferencia de ROC-AUC es de apenas **0.007** y su intervalo al 95 %
  incluye el cero, así que **no se puede afirmar que un modelo sea superior**.
* Las variables con mayor contribución son el PIB per cápita, la duración y
  la población, seguidas por el tipo de desastre y la región.
* Ambos modelos se exportan a la aplicación y se promedian para dar una
  estimación más estable.